# Moments and timescales

Section 1.3 of the lecture notes, at the keyboard: the moments and the timescales of §1.3.4.  We
take one two-type Hawkes process with an exponential kernel, in three parametrisations, and set
each formula of the section against a Monte Carlo estimate with its band of two standard errors.

Stationary quantities — the mean, the covariance, the intensity read at the events — are
estimated along one long path.  The forward mean and the mean response describe the process
started from a given state, and are estimated across *fans*: many independent paths started from
that state.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from unito26.lob import config
from unito26.lob.hawkes import (
    Clock, ExponentialHawkes, HawkesParams, decayed_counts_at, intensities_at_events,
)
from unito26.lob.visualization import (
    INK, MUTED, PALETTE, monte_carlo_figure, raster_figure, use_template,
)

use_template()

PAIRS = {
    "self-exciting": config.self_exciting_pair_params(),
    "cross-exciting": config.cross_exciting_pair_params(),
    "asymmetric": config.asymmetric_pair_params(),
}
TYPES = ("type 1", "type 2")


def show(vector):
    return "(" + ", ".join(f"{value:.4g}" for value in vector) + ")"


print(pd.DataFrame([
    {
        "pair": name,
        "beta (1/s)": params.decay,
        "mu": show(params.baseline),
        "lambda*": show(params.stationary_intensity()),
        "column sums of Gamma": show(params.branching_matrix.sum(axis=0)),
        "rho": round(params.branching_ratio, 4),
        "m = lambda*/beta": show(params.stationary_intensity() / params.decay),
        "relaxation time (s)": round(params.relaxation_time(), 4),
    }
    for name, params in PAIRS.items()
]).to_string(index=False))

The first two parametrisations share $\lambda^*$ and $\rho$ and differ only in who excites whom:
in the first each type mostly excites itself, in the second mostly the other.  Their column sums of
$\Gamma$ are constant; those of the third are not.  Types are indexed from zero in the code and
named 1 and 2 in the text, and in printed output and figures a prime is a transpose.

## 1. The first moment

Theorem 1.3.19 gives the stationary intensity, $\lambda^* = (I - \Gamma)^{-1}\mu$, and its
estimator on one path is the count per unit time, $N_e(T)/T$, taken after a burn-in that lets the
path forget its empty start.  A time average along one path estimates a stationary quantity because
the process is ergodic, which §1.3.4.4 lists among the things that need $\rho < 1$.

What needs care is the standard error.  $\sqrt{\lambda^*_e/T}$ is right for a Poisson process,
whose counts in disjoint windows are independent, and a Hawkes process is not one.  Batch means
assume less: cut the path into $B$ contiguous batches of length $L$, long against the relaxation
time, and read the standard error of the mean off the spread of the $B$ batch means.

In [ ]:
BURN_IN = 50.0
HORIZON = 20_000.0
BATCHES = 50


def path(simulator, horizon):
    '''The events of (0, horizon] as two arrays: the times, and the types.'''
    events = list(simulator.events(horizon))
    times = np.array([time for time, _ in events], dtype=float)
    types = np.array([kind for _, kind in events], dtype=int)
    return times, types


def batch_means(series):
    '''The mean of a stationary series along its first axis, and its standard error from
    the spread of the means of BATCHES contiguous batches.'''
    series = np.asarray(series, dtype=float)
    size = len(series) // BATCHES
    means = series[: BATCHES * size].reshape(BATCHES, size, *series.shape[1:]).mean(axis=1)
    return means.mean(axis=0), means.std(axis=0, ddof=1) / np.sqrt(BATCHES)


long_paths = {
    name: path(ExponentialHawkes(params, seed), BURN_IN + HORIZON)
    for (name, params), seed in zip(PAIRS.items(), (11, 12, 13))
}

seconds = np.arange(BURN_IN, BURN_IN + HORIZON + 1.0)
rows = []
for name, params in PAIRS.items():
    times, types = long_paths[name]
    counts = np.stack([np.histogram(times[types == e], seconds)[0] for e in range(2)], axis=1)
    rate, error = batch_means(counts)
    poisson = np.sqrt(rate / HORIZON)
    for e, label in enumerate(TYPES):
        stationary = params.stationary_intensity()[e]
        rows.append({
            "pair": name, "type": label, "lambda*": stationary, "N/T": rate[e],
            "SE, batch means": error[e], "SE, Poisson": poisson[e],
            "ratio": error[e] / poisson[e], "z": (rate[e] - stationary) / error[e],
        })
print(pd.DataFrame(rows).round(4).to_string(index=False))

$z$ is the difference from the formula in units of the standard error, here the batch-means one.
Where the formula holds, $z$ is close to a standard normal draw, so about one comparison in
twenty falls beyond two standard errors.  On these seeds every $N_e(T)/T$ is within two standard
errors of $\lambda^*_e$.

The squared ratio of the two errors is a Fano factor.  With $s^2$ the sample variance of the batch
counts $N_b$ and $\hat\lambda = N_e(T)/T$,

$$\frac{\mathrm{SE}^2_{\text{batch}}}{\mathrm{SE}^2_{\text{Poisson}}}
= \frac{s^2/(L^2B)}{\hat\lambda/(LB)} = \frac{s^2}{\hat\lambda L},$$

the Fano factor of the count in one batch, which is one for a Poisson process; for batches long
against the relaxation time it is at its plateau, the value it reaches over long windows.  The
Poisson error understates the batch-means one by its square root.  The batch rates of one type,
against the band a Poisson count would keep them in:

In [ ]:
params = PAIRS["self-exciting"]
times, types = long_paths["self-exciting"]
edges = np.linspace(BURN_IN, BURN_IN + HORIZON, BATCHES + 1)
length = HORIZON / BATCHES
batch_rates = np.histogram(times[types == 0], edges)[0] / length
level = params.stationary_intensity()[0]
spread = 2 * np.sqrt(level / length)
print(f"batch rates outside the Poisson band: {np.sum(np.abs(batch_rates - level) > spread)}"
      f" of {BATCHES}")

figure = go.Figure()
figure.add_hrect(y0=level - spread, y1=level + spread, fillcolor=MUTED, opacity=0.2, line_width=0)
figure.add_hline(y=level, line=dict(color=INK, width=1))
figure.add_scatter(x=(edges[:-1] + edges[1:]) / 2, y=batch_rates, mode="markers",
                   name="rate of type 1 in one batch", marker=dict(color=PALETTE[0], size=7))
figure.update_layout(
    title="Self-exciting pair, type 1: batch rates against the ±2 SD band of a Poisson count",
    height=380, legend=dict(orientation="h", y=1.1, x=0),
)
figure.update_xaxes(title_text="time (s)")
figure.update_yaxes(title_text="events per second")
figure

A Poisson process would leave about one batch rate in twenty outside the band.  The clusters make
the count in a batch vary more than a Poisson count would, and the Fano factor is the ratio of the
two variances.

The state gives no second estimator of the first moment.  From the proof of
Proposition 1.3.12, on every path,

$$\int_0^T Z_e(u)\,du = \frac{N_e(T-) - Z_e(T)}{\beta},$$

so the time average of $Z_e$ is $N_e(T)/(\beta T)$ up to a term of order $1/T$: the count divided
by $\beta$.  It is an identity of the path, not a statistical statement, and setting the time
average of $Z$ against $m = \lambda^*/\beta$ of Proposition 1.3.21 tests nothing that
$N_e(T)/T$ has not already tested.  On the whole asymmetric path, burn-in included, with the
integral taken one stretch between events at a time:

In [ ]:
def integral_of_state(decay, times, types, horizon):
    '''The integral of Z over (0, horizon], one stretch between events at a time: from the
    state z, a stretch of length s contributes z (1 - exp(-beta s)) / beta.'''
    state = np.zeros(2)
    integral = np.zeros(2)
    last = 0.0
    for time, kind in zip(times, types):
        integral += state * (1.0 - np.exp(-decay * (time - last))) / decay
        state = state * np.exp(-decay * (time - last))
        state[kind] += 1.0
        last = time
    return integral + state * (1.0 - np.exp(-decay * (horizon - last))) / decay


params = PAIRS["asymmetric"]
times, types = long_paths["asymmetric"]
end = BURN_IN + HORIZON
counts = np.bincount(types, minlength=2)
state_at_end = decayed_counts_at(
    params.decay, times, types, np.zeros(2), Clock(np.array([end]))
)[0]

by_stretches = integral_of_state(params.decay, times, types, end)
closed_form = (counts - state_at_end) / params.decay
print("integral of Z, by stretches :", by_stretches)
print("(N(T-) - Z(T)) / beta        :", closed_form)
print("largest relative difference  :", np.max(np.abs(by_stretches / closed_form - 1)))
print()
print("time average of Z            :", by_stretches / end)
print("N(T) / (beta T)              :", counts / (params.decay * end))

The integral and its closed form agree to rounding, and the time average falls short of
$N_e(T)/(\beta T)$ by $Z_e(T)/(\beta T)$.  An estimate of $m$ that is not the count in disguise
needs independent paths: $Z$ read at one fixed time on each of many paths, each started empty and
run for twenty relaxation times.  The last column is what remains of the empty start at that time,
$m_t(0) - m = -e^{Kt}m$, from Proposition 1.3.33, which section 4 takes up.

In [ ]:
PATHS_AT_ONE_TIME = 4_000


def state_after(params, moment, rng):
    '''Z(moment) on one path started empty.'''
    times, types = path(ExponentialHawkes(params, rng), moment)
    clock = Clock(np.array([moment]))
    return decayed_counts_at(params.decay, times, types, np.zeros(2), clock)[0]


rows = []
for (name, params), seed in zip(PAIRS.items(), (21, 22, 23)):
    rng = np.random.default_rng(seed)
    moment = 20 * params.relaxation_time()
    readings = np.array([state_after(params, moment, rng) for _ in range(PATHS_AT_ONE_TIME)])
    mean = readings.mean(axis=0)
    error = readings.std(axis=0, ddof=1) / np.sqrt(PATHS_AT_ONE_TIME)
    m = params.stationary_intensity() / params.decay
    memory = params.forward_mean_state(np.zeros(2), moment) - m
    for e, label in enumerate(TYPES):
        rows.append({
            "pair": name, "type": label, "m": m[e], "mean over paths": mean[e], "SE": error[e],
            "z": (mean[e] - m[e]) / error[e], "memory of the start": f"{memory[e]:.1e}",
        })
print(pd.DataFrame(rows).round(4).to_string(index=False))

On these seeds the mean state across independent paths is $m = \lambda^*/\beta$ within two
standard errors, on every pair and both types.

## 2. The second moment

Theorem 1.3.29 gives the stationary covariance of the state, $V$, as the solution of the Lyapunov
equation (1.3.26),

$$KV + VK^\top + \operatorname{diag}(\lambda^*) = 0, \qquad K = A - \beta I,$$

and the stationary covariance of the intensity as $AVA^\top$, since $\lambda = \mu + AZ$.
`stationary_covariance` solves the equation.  Along each long path we read $Z$ on a regular clock
after the burn-in, and set the covariance of the readings against $V$, with batch-means errors.
The standard deviation of $\lambda_e$ is the square root of a variance, and its standard error is
the variance's divided by twice the standard deviation.

In [ ]:
STEP = 0.5
clock = Clock(np.arange(BURN_IN, BURN_IN + HORIZON, STEP))
states = {}
for name, params in PAIRS.items():
    times, types = long_paths[name]
    states[name] = decayed_counts_at(params.decay, times, types, np.zeros(2), clock)

rows = []
for name, params in PAIRS.items():
    V = params.stationary_covariance()
    centred = states[name] - states[name].mean(axis=0)
    for i, j in ((0, 0), (0, 1), (1, 1)):
        estimate, error = batch_means(centred[:, i] * centred[:, j])
        rows.append({"pair": name, "quantity": f"V[{i + 1},{j + 1}]", "formula": V[i, j],
                     "one path": estimate, "SE": error, "z": (estimate - V[i, j]) / error})
    intensities = params.baseline + states[name] @ params.excitation.T
    variance, error = batch_means((intensities - intensities.mean(axis=0)) ** 2)
    formula = np.sqrt(np.diag(params.excitation @ V @ params.excitation.T))
    for e, label in enumerate(TYPES):
        deviation = np.sqrt(variance[e])
        rows.append({"pair": name, "quantity": f"SD of lambda, {label}", "formula": formula[e],
                     "one path": deviation, "SE": error[e] / (2 * deviation),
                     "z": (deviation - formula[e]) / (error[e] / (2 * deviation))})
print(pd.DataFrame(rows).round(4).to_string(index=False))

On these seeds every entry of $V$ and every standard deviation of the intensity is within two
standard errors of its estimate.  The two symmetric pairs have the same $\lambda^*$ and the same
$\rho$, and a different $V$: the covariance of the two types is small where each type excites
itself and large where each excites the other.

When $A$ has constant column sums, contracting (1.3.26) with $\mathbf 1$ on both sides gives the
scalar formula (1.3.29),

$$\mathbf 1^\top V\mathbf 1 = \frac{\bar\lambda}{2\beta(1-\rho)} .$$

Remark 1.3.32 gives its scope: it concerns the total decayed count, and it needs the constant
column sums.  The symmetric pairs have them and the asymmetric pair does not.

In [ ]:
rows = []
for name, params in PAIRS.items():
    total = states[name].sum(axis=1)
    estimate, error = batch_means((total - total.mean()) ** 2)
    lyapunov = params.stationary_covariance().sum()
    scalar = params.stationary_intensity().sum() / (2 * params.decay * (1 - params.branching_ratio))
    rows.append({
        "pair": name, "column sums of A": show(params.excitation.sum(axis=0)),
        "1'V1": lyapunov, "(1.3.29)": scalar, "one path": estimate, "SE": error,
        "z against 1'V1": (estimate - lyapunov) / error,
        "z against (1.3.29)": (estimate - scalar) / error,
    })
scalar_case = pd.DataFrame(rows)
print(scalar_case.round(4).to_string(index=False))

figure = go.Figure()
figure.add_scatter(
    x=scalar_case["pair"], y=scalar_case["one path"], mode="markers", name="one path, ±2 SE",
    error_y=dict(array=2 * scalar_case["SE"], color=PALETTE[0], thickness=1.5, width=6),
    marker=dict(color=PALETTE[0], size=9),
)
figure.add_scatter(
    x=scalar_case["pair"], y=scalar_case["1'V1"], mode="markers", name="1'V1, Lyapunov",
    marker=dict(symbol="line-ew-open", size=34, color=INK, line=dict(width=2)),
)
figure.add_scatter(
    x=scalar_case["pair"], y=scalar_case["(1.3.29)"], mode="markers", name="scalar formula",
    marker=dict(symbol="x-thin-open", size=14, color=PALETTE[1], line=dict(width=2)),
)
figure.update_layout(title="Variance of the total state 1'Z: one path, Lyapunov, (1.3.29)",
                     height=400, legend=dict(orientation="h", y=1.1, x=0))
figure.update_yaxes(title_text="variance")
figure

On the symmetric pairs the scalar formula is the Lyapunov value, and the estimate is within two
standard errors of it.  On the asymmetric pair the estimate is within two standard errors of the
Lyapunov value and many standard errors above the scalar formula.

## 3. The Palm trap

The mean of the intensity over the events is not its mean over time.  Take
$Y = \lambda_e\mathbf 1_{(0,T]}$ in (1.3.5): it is non-negative and predictable.  Summing (1.3.5)
over the types, with $\lambda_{\mathfrak g} = \mathbf 1^\top\lambda$ the intensity of the ground
process, and using $\operatorname{Cov}(\lambda) = AVA^\top$ in the stationary version,

$$\mathbb E\Big[\sum_{n:\,T_n\le T}\lambda_e(T_n)\Big]
= \mathbb E\Big[\int_0^T\lambda_e(t)\,\lambda_{\mathfrak g}(t)\,dt\Big]
= T\big((AVA^\top\mathbf 1)_e + \lambda^*_e\bar\lambda\big).$$

Dividing by $\mathbb E[N_{\mathfrak g}(T)] = \bar\lambda T$, the long-run mean of $\lambda_e$ over
the events is

$$\lambda^*_e + \frac{(AVA^\top\mathbf 1)_e}{\bar\lambda} .$$

The intensity is left-continuous, so $\lambda_e(T_n)$ is the value before the jump.
Read after the jump it gains $Ae_{E_n}$ at each event, and these gains average to
$A\lambda^*/\bar\lambda$.  `intensities_at_events` replays the path and reads $\lambda$ before each
jump.

In [ ]:
at_events = {}
rows = []
for name, params in PAIRS.items():
    times, types = long_paths[name]
    after = times > BURN_IN
    stationary = params.stationary_intensity()
    total = stationary.sum()
    covariance = params.excitation @ params.stationary_covariance() @ params.excitation.T
    pre_jump = intensities_at_events(params, times, types)[after]
    at_events[name] = pre_jump
    readings = {
        "on the clock": (params.baseline + states[name] @ params.excitation.T, stationary),
        "at events, before the jump": (pre_jump, stationary + covariance.sum(axis=1) / total),
        "at events, after the jump": (
            pre_jump + params.excitation[:, types[after]].T,
            stationary + (covariance.sum(axis=1) + params.excitation @ stationary) / total,
        ),
    }
    for reading, (sample, formula) in readings.items():
        estimate, error = batch_means(sample)
        for e, label in enumerate(TYPES):
            rows.append({"pair": name, "type": label, "reading": reading, "formula": formula[e],
                         "one path": estimate[e], "SE": error[e],
                         "z": (estimate[e] - formula[e]) / error[e]})
print(pd.DataFrame(rows).round(4).to_string(index=False))

On the clock the intensity averages $\lambda^*$; at the events it averages well above it, and on
these seeds the formula accounts for the difference within two standard errors.  Neither estimate
is wrong.  A mean over the events is a mean under the Palm distribution, the law of the process
seen from a typical event, and a typical event arrives where the intensity is high.

In [ ]:
params = PAIRS["self-exciting"]
samples = {
    "on the clock": params.baseline[0] + states["self-exciting"] @ params.excitation[0],
    "at the events, before the jump": at_events["self-exciting"][:, 0],
}
levels = np.geomspace(params.baseline[0], max(sample.max() for sample in samples.values()), 400)
figure = go.Figure()
for (label, sample), colour in zip(samples.items(), PALETTE):
    above = 1.0 - np.searchsorted(np.sort(sample), levels, side="right") / len(sample)
    figure.add_scatter(x=levels, y=above, mode="lines", name=label,
                       line=dict(color=colour, width=2))
    figure.add_vline(x=sample.mean(), line=dict(color=colour, width=1, dash="dash"))
figure.update_layout(
    title="Self-exciting pair: share of the readings of λ₁ above x, with the means dashed",
    height=400, legend=dict(orientation="h", y=1.1, x=0),
)
figure.update_xaxes(title_text="x (1/s)", type="log")
figure.update_yaxes(title_text="share of the readings above x", range=[0, 1])
figure

On the clock the intensity of type 1 spends much of the time near its baseline; at the events most
of its readings lie far above it.  The overstatement is a covariance divided by a rate, large
where the intensity is volatile and the flow thin.  A fitted intensity averaged over the observed
events is a Palm average, and reads high by this term.

## 4. The forward mean

Proposition 1.3.33 gives the mean of the state and of the counts at a later time, given the state
after a time $s$.  With $m_{s,t}(z) = \mathbb E[Z(t) \mid Z(s+) = z]$ and $h = t - s$,

$$m_{s,t}(z) = e^{Kh}z + \Phi(h)\mu,
\qquad
\mathbb E[N(t) - N(s) \mid Z(s+) = z] = h\mu + A\Phi(h)z + A\Big(\int_0^h\Phi(u)\,du\Big)\mu,$$

with $\Phi(h) = \int_0^h e^{Ku}\,du$.  Both depend on $s$ and $t$ through $h$ alone.  We move the
clock to $s = 0$, and write $m_t(z)$ for $m_{0,t}(z)$ and $\mathbb E_z N(h)$ for the expected count
over $(0, h]$.  When $\rho < 1$ both read more simply as deviations from the stationary mean,
(1.3.32) and (1.3.33).  By Proposition 1.3.21, $\mu = -Km$, and $\Phi(t) = K^{-1}(e^{Kt} - I)$
commutes with $K$, so $\Phi(t)\mu = (I - e^{Kt})m$ and

$$m_t(z) - m = e^{Kt}(z - m) .$$

Integrating $\mu + Am_s(z)$ over $(0, h]$, as the proof does, and using $\mu + Am = \lambda^*$,

$$\mathbb E_z N(h) = h\lambda^* + A\Phi(h)(z - m) .$$

The state forgets its start through $e^{Kt}$ alone, and the start shifts the expected count by a
term that stays bounded in $h$.  The two forms agree to rounding:

In [ ]:
asymmetric = PAIRS["asymmetric"]
stationary = asymmetric.stationary_intensity()
m = stationary / asymmetric.decay


def mean_state_as_deviation(params, state, t):
    '''m + e^{Kt} (z - m).'''
    m = params.stationary_intensity() / params.decay
    return m + params.mean_response(t) @ (state - m)


def mean_counts_as_deviation(params, state, t):
    '''t lambda* + A Phi(t) (z - m).'''
    m = params.stationary_intensity() / params.decay
    excess = params.excitation @ params.mean_response_integral(t) @ (state - m)
    return t * params.stationary_intensity() + excess


state_gaps, count_gaps = [], []
for z in (np.zeros(2), m + (0.0, 1.0), np.array([0.0, 3.0])):
    for t in (0.1, 0.5, 2.0, 10.0):
        notes = asymmetric.forward_mean_state(z, t), asymmetric.forward_mean_counts(z, t)
        state_gaps.append(np.abs(notes[0] - mean_state_as_deviation(asymmetric, z, t)).max())
        count_gaps.append(np.abs(notes[1] - mean_counts_as_deviation(asymmetric, z, t)).max())
print("largest difference, mean state :", max(state_gaps))
print("largest difference, mean counts:", max(count_gaps))

A fan is a set of independent paths started from one state and read on a common clock.  Its mean at
each time estimates $m_t(z)$, or $\mathbb E_z N(t)$, and since the paths are independent the
standard error is the spread across paths over the square root of their number.  Neighbouring times
share their paths, so their errors are correlated: where a formula leaves the band of two standard
errors it does so for a run of points and comes back, and a few such runs on a curve of many points
are what agreement looks like.  A formula wrong by more than the noise leaves the band and stays
out.

We run four fans on the asymmetric pair: from $m$; from $m + e_1$ and from $m + e_2$, which carry
one extra event of either type; and from $z = (0, 3)$, as if three events of type 2 had arrived at
time zero and nothing else ever had.

In [ ]:
PATHS = 20_000
CLOCK = Clock(np.linspace(0.0, 3.0, 61))
FAR = np.array([0.0, 3.0])


def fan(params, state, clock, paths, rng):
    '''Z and N read on the clock, on independent paths continued from Z(0+) = state; N
    counts the events after 0.  Both arrays are (paths, len(clock), d).'''
    states = np.empty((paths, len(clock), params.dimension))
    counts = np.empty((paths, len(clock), params.dimension))
    for k in range(paths):
        times, types = path(ExponentialHawkes.from_state(params, state, rng), clock[-1])
        states[k] = decayed_counts_at(params.decay, times, types, state, clock)
        for e in range(params.dimension):
            counts[k, :, e] = np.searchsorted(times[types == e], clock, side="right")
    return states, counts


def outside_band(samples, formula):
    '''How many points of the formula after the start lie outside the band of two
    standard errors about the Monte Carlo mean, and out of how many, as a phrase.'''
    mean = samples.mean(axis=0)[1:]
    error = samples.std(axis=0, ddof=1)[1:] / np.sqrt(len(samples))
    outside = int(np.sum(np.abs(mean - formula[1:]) > 2 * error))
    return f"outside the band at {outside} of {mean.size} points"


def forward_states(params, state, clock):
    return np.array([params.forward_mean_state(state, t) for t in clock])


def forward_counts(params, state, clock):
    return np.array([params.forward_mean_counts(state, t) for t in clock])


starts = {"m": m, "m + e1": m + (1.0, 0.0), "m + e2": m + (0.0, 1.0), "far": FAR}
fans = {
    label: fan(asymmetric, start, CLOCK, PATHS, np.random.default_rng(seed))
    for (label, start), seed in zip(starts.items(), (41, 42, 43, 44))
}
events = sum(int(counts[:, -1].sum()) for _, counts in fans.values())
print(f"{len(fans)} fans of {PATHS:,} paths, {events:,} events in all")

In [ ]:
for label in ("m + e1", "m + e2"):
    formula = forward_states(asymmetric, starts[label], CLOCK)
    print(f"mean state from {label}: formula", outside_band(fans[label][0], formula))

formula = forward_states(asymmetric, starts["m + e2"], CLOCK)

figure = monte_carlo_figure(CLOCK, fans["m + e2"][0], formula, TYPES,
                            "Mean state from m + e2: fan against (1.3.30)")
figure.update_xaxes(title_text="t (s)")
figure.update_yaxes(title_text="mean of Z(t)")
figure

The extra event of type 2 raises $Z_2$ by one at time zero, and that unit decays.  $Z_1$ starts at
its stationary mean and rises before it returns, because type 2 excites type 1 strongly.  The start
far from $m$ shows the same thing at a larger scale; forty of its paths are drawn in grey.

In [ ]:
far_states = fans["far"][0]
formula = forward_states(asymmetric, FAR, CLOCK)
error = far_states[:, :, 0].std(axis=0, ddof=1) / np.sqrt(PATHS)
peak = formula[:, 0].argmax()
print(f"mean of Z_1: {formula[peak, 0]:.3f} at t = {CLOCK[peak]:.2f} s, against m_1 = {m[0]:.3f}")
print("mean state of type 1: formula", outside_band(far_states[:, :, :1], formula[:, :1]))

figure = go.Figure()
for sample in far_states[:40, :, 0]:
    figure.add_scatter(x=CLOCK, y=sample, mode="lines", line=dict(color=MUTED, width=0.6),
                       opacity=0.4, showlegend=False, hoverinfo="skip")
figure.add_scatter(x=CLOCK, y=far_states[:, :, 0].mean(axis=0), mode="markers",
                   name="type 1, fan mean ±2 SE", marker=dict(color=PALETTE[0], size=5),
                   error_y=dict(array=2 * error, color=PALETTE[0], thickness=1, width=0))
figure.add_scatter(x=CLOCK, y=formula[:, 0], mode="lines", name="type 1, formula",
                   line=dict(color=PALETTE[0], width=2))
figure.add_hline(y=m[0], line=dict(color=INK, width=1, dash="dot"),
                 annotation_text="stationary mean of Z_1", annotation_position="top right")
figure.update_layout(title="From z = (0, 3): the state of type 1 on forty paths, and its mean",
                     height=440, legend=dict(orientation="h", y=1.1, x=0))
figure.update_xaxes(title_text="t (s)")
figure.update_yaxes(title_text="Z_1(t)")
figure

$Z_1$ starts below its stationary mean and its mean rises well above it before relaxing.  The
deviation evolves by $e^{Kt}$, whose entries need not be monotone, so the mean of one coordinate can
overshoot its stationary value on the way to it.

The counts at a few horizons against (1.3.31), checked on every fan and drawn for the far start,
with $h\lambda^*$ dotted:

In [ ]:
COARSE = slice(0, None, 5)
for label, start in starts.items():
    formula = forward_counts(asymmetric, start, CLOCK)[COARSE]
    print(f"counts from {label:<6}: formula", outside_band(fans[label][1][:, COARSE], formula))

formula = forward_counts(asymmetric, FAR, CLOCK)

figure = monte_carlo_figure(CLOCK[COARSE], fans["far"][1][:, COARSE], formula[COARSE], TYPES,
                            "Expected counts from z = (0, 3): fan against (1.3.31)")
for e, label in enumerate(TYPES):
    figure.add_scatter(x=CLOCK, y=CLOCK * stationary[e], mode="lines", name=f"{label}, h lambda*",
                       line=dict(color=PALETTE[e], width=1, dash="dot"))
figure.update_xaxes(title_text="h (s)")
figure.update_yaxes(title_text="mean of N(h)")
figure

Both counts run above $h\lambda^*$, and the gap settles to a constant as $h$ grows: the term
$A\Phi(h)(z - m)$ is bounded.

From $z = m$ the mean does not move, $m_t(m) = m$, but the fan still spreads.  In the proof of
Theorem 1.3.29, hold $m_t = m$ and write the second moment as $C_t + mm^\top$: the cross terms
cancel exactly as they do there, and the covariance of the fan solves
$\dot C_t = KC_t + C_tK^\top + \operatorname{diag}(\lambda^*)$ with $C_0 = 0$, the start being
deterministic.  Hence

$$C_t = \int_0^t e^{Ks}\operatorname{diag}(\lambda^*)\,e^{K^\top s}\,ds = V - e^{Kt}Ve^{K^\top t},$$

which grows from zero to the $V$ of section 2.  The mean of $(Z_e(t) - m_e)^2$ across the fan
estimates the diagonal of $C_t$, and $m$ being exact, it needs no correction for an estimated
mean.

In [ ]:
V = asymmetric.stationary_covariance()
from_m = fans["m"][0]
print("mean of the fan from m: m", outside_band(from_m, np.broadcast_to(m, from_m.shape[1:])))

spread = np.array([np.diag(V - asymmetric.mean_response(t) @ V @ asymmetric.mean_response(t).T)
                   for t in CLOCK])
print("variance of the fan from m: formula", outside_band((from_m - m) ** 2, spread))

figure = monte_carlo_figure(CLOCK, (from_m - m) ** 2, spread, TYPES,
                            "Variance of the fan from m against V - exp(Kt) V exp(K't)")
for e in range(2):
    figure.add_hline(y=V[e, e], line=dict(color=PALETTE[e], width=1, dash="dot"))
figure.update_xaxes(title_text="t (s)")
figure.update_yaxes(title_text="variance of Z(t)")
figure

The fan from $m$ keeps its mean at $m$ and widens from nothing to the stationary spread, dotted,
along the formula.  Where its points leave the band they do so in a run and come back.

## 5. The mean response

Corollary 1.3.35 calls mean response the displacement of the mean state by one more event of type
$j$ in the state, $R_j(h) = m_{s,s+h}(z + e_j) - m_{s,s+h}(z)$, which depends neither on $s$ nor on
$z$.  The fans from $m + e_j$ and from $m$ estimate the two means, and the difference of their
means estimates

$$R_j(t) = e^{Kt}e_j .$$

The fans are independent, so the standard error of the difference is the root of the sum of their
squared standard errors.  Pairing path $k$ of one fan with path $k$ of the other, which is
arbitrary, and taking the spread of the differences computes exactly that.

In [ ]:
difference = {j: fans[f"m + e{j + 1}"][0] - fans["m"][0] for j in range(2)}
EXTRA = ("extra event of type 1", "extra event of type 2")

total_state = np.stack([difference[j].sum(axis=2) for j in range(2)], axis=2)
formula = np.array([asymmetric.mean_response(t).sum(axis=0) for t in CLOCK])
print("formula", outside_band(total_state, formula))

figure = monte_carlo_figure(CLOCK, total_state, formula, EXTRA,
                            "Response of the total mean state, 1'R_j(t): fans against (1.3.34)")
figure.add_hline(y=1.0, line=dict(color=INK, width=1, dash="dot"))
figure.update_xaxes(title_text="t (s)")
figure.update_yaxes(title_text="1'R_j(t)")
figure

By Remark 1.3.37,

$$\frac{d}{dt}\Big|_{t=0}\mathbf 1^\top R_j(t) = \beta\big((\mathbf 1^\top\Gamma)_j - 1\big),$$

so the response of the total state to an event of type $j$ rises before it falls exactly when the
$j$-th column sum of $\Gamma$ exceeds one.  Type 2's does and type 1's does not.

In [ ]:
column_sums = asymmetric.branching_matrix.sum(axis=0)
print("column sums of Gamma      :", column_sums)
print("initial slopes of 1'R_j   :", asymmetric.decay * (column_sums - 1))

fine = np.linspace(0.0, 1.0, 1001)
curve = np.array([asymmetric.mean_response(t)[:, 1].sum() for t in fine])
print(f"peak of 1'R_2             : {curve.max():.4f} at t = {fine[curve.argmax()]:.3f} s")

That is the response of the total mean *state*.  The response of the total mean *intensity*,
$\mathbf 1^\top AR_j(t)$, weights each type by its column sum of $A$, the jump it causes in
$\lambda_{\mathfrak g}$, and it falls from the start for both types.

In [ ]:
total_intensity = np.stack(
    [(difference[j] @ asymmetric.excitation.T).sum(axis=2) for j in range(2)], axis=2
)
formula = np.array([(asymmetric.excitation @ asymmetric.mean_response(t)).sum(axis=0)
                    for t in CLOCK])
print("column sums of A          :", asymmetric.excitation.sum(axis=0))
wide = np.linspace(0.0, 20.0, 2001)
falling = np.array([(asymmetric.excitation @ asymmetric.mean_response(t)).sum(axis=0)
                    for t in wide])
print("1'A R_j decreasing on [0, 20] s:", np.all(np.diff(falling, axis=0) < 0, axis=0))
print("formula", outside_band(total_intensity, formula))

figure = monte_carlo_figure(CLOCK, total_intensity, formula, EXTRA,
                            "Response of the total mean intensity, 1'A R_j(t)")
figure.update_xaxes(title_text="t (s)")
figure.update_yaxes(title_text="1'A R_j(t) (1/s)")
figure

An event of type 2 has more than one direct offspring on average, mostly of type 1, and a unit of
state of type 1 moves the total intensity by much less than a unit of type 2 does.  So the total
state gains while the total intensity loses, and a statement about the response to an event has to
name the quantity that responds.

The counts carry the response too.  By (1.3.33) the two fans' expected counts
differ by $A\Phi(h)e_j$, and by (1.3.35) its total tends to $C_j - 1$, where
$C_j = \mathbf 1^\top(I - \Gamma)^{-1}e_j$ is the expected size of a cluster founded by an event of
type $j$, Proposition 1.3.23.

In [ ]:
count_difference = np.stack(
    [(fans[f"m + e{j + 1}"][1] - fans["m"][1]).sum(axis=2) for j in range(2)], axis=2
)
formula = np.array([(asymmetric.excitation @ asymmetric.mean_response_integral(t)).sum(axis=0)
                    for t in CLOCK])
cluster_sizes = np.linalg.solve((np.eye(2) - asymmetric.branching_matrix).T, np.ones(2))
print("C_j - 1                   :", cluster_sizes - 1)
print("formula at the last h     :", formula[-1])
print("formula", outside_band(count_difference[:, COARSE], formula[COARSE]))

figure = monte_carlo_figure(CLOCK[COARSE], count_difference[:, COARSE], formula[COARSE], EXTRA,
                            "Extra events counted by h: fans against 1'A Φ(h) e_j")
for j in range(2):
    figure.add_hline(y=cluster_sizes[j] - 1, line=dict(color=PALETTE[j], width=1, dash="dot"))
figure.update_xaxes(title_text="h (s)")
figure.update_yaxes(title_text="difference of the mean total counts")
figure

The limit is $C_j - 1$ and not $C_j$: the extra event sits in the state at time zero, $N$ counts
the events of $(0, h]$, and what the extra event adds to the count is its descendants.

The band widens with $h$, since both counts accumulate variance.  The two fans are independent;
seeding path $k$ of both alike would correlate them, but only in part, since the extra event
changes the waiting times from the first draw on.  The exact coupling is a ghost cluster: by the
construction of Proposition 1.3.8, the copy with the extra event is the copy without it together
with the independent descendants of that event, so the difference of the two is the state of that
cluster alone.  Writing it is left as an exercise.

## 6. The relaxation time, and the dimensionless groups

Definition 1.3.36 calls $1/(\beta(1-\rho))$ the relaxation time, and Remark 1.3.37 warns that it
describes the response only asymptotically.  One projection of the mean follows it at every time.
Let $v$ be a left Perron vector of $\Gamma$, so that $v^\top K = -\beta(1-\rho)\,v^\top$.
Contracting (1.3.32) with $v$,

$$v^\top\big(m_t(z) - m\big) = e^{-\beta(1-\rho)t}\,v^\top(z - m),$$

exactly and from $t = 0$, whatever $z$ is.  On the symmetric pairs $v \propto \mathbf 1$.

In [ ]:
values, vectors = np.linalg.eig(asymmetric.branching_matrix.T)
v = np.real(vectors[:, np.argmax(values.real)])
v = v / v.sum()
print("left Perron vector v      :", v)
print("v' Gamma / v'             :", (v @ asymmetric.branching_matrix) / v)
print(f"relaxation time           : {asymmetric.relaxation_time():.4f} s")

SHOWN = CLOCK <= 2.0
projected = ("far", "m + e2")
projections = np.stack([(fans[label][0] - m) @ v for label in projected], axis=2)
formula = np.stack([np.exp(-CLOCK / asymmetric.relaxation_time()) * (v @ (starts[label] - m))
                    for label in projected], axis=1)
print("formula", outside_band(projections[:, SHOWN], formula[SHOWN]))

figure = monte_carlo_figure(CLOCK[SHOWN], projections[:, SHOWN], formula[SHOWN],
                            ("from z = (0, 3)", "from m + e2"),
                            "Perron projection of the mean state, v'(Z(t) - m), on a log axis")
figure.update_xaxes(title_text="t (s)")
figure.update_yaxes(title_text="v'(Z(t) - m)", type="log")
figure

The fan whose $Z_1$ overshoots and the fan whose total state rises before it falls both project
onto straight lines on a log axis, parallel, decaying at the rate $1/\tau$, with $\tau$ the
relaxation time.

A start with no Perron component is the other side of the same statement: its mean never relaxes
at the relaxation rate.  On the symmetric pairs $v \propto \mathbf 1$, and $z - m \propto (1, -1)$
is an eigenvector of $\Gamma$ for its second eigenvalue $\gamma_2$.  Then $v^\top(z - m) = 0$ and

$$m_t(z) - m = e^{-\beta(1-\gamma_2)t}(z - m),$$

one exponential, faster than $\beta(1-\rho)$.  The start must be non-negative, which bounds the
deviation by $m$; $z = (1/2, 0)$ is the largest.

In [ ]:
CONTRAST = np.array([0.5, 0.0])
CONTRAST_CLOCK = Clock(np.linspace(0.0, 2.0, 41))
SYMMETRIC = ("self-exciting", "cross-exciting")

for name in SYMMETRIC:
    params = PAIRS[name]
    deviation = CONTRAST - params.stationary_intensity() / params.decay
    ratio = (params.branching_matrix @ deviation) / deviation
    print(f"{name:<15} z - m = {show(deviation)}   Gamma (z - m) / (z - m) = {show(ratio)}   "
          f"rate {params.decay * (1 - ratio[0]):.2f} 1/s, relaxation rate "
          f"{1 / params.relaxation_time():.2f} 1/s")

contrast_fans = {
    name: fan(PAIRS[name], CONTRAST, CONTRAST_CLOCK, PATHS, np.random.default_rng(seed))[0]
    for name, seed in zip(SYMMETRIC, (61, 62))
}

In [ ]:
contrast = np.stack([contrast_fans[name][:, :, 0] - contrast_fans[name][:, :, 1]
                     for name in SYMMETRIC], axis=2)
formula = np.stack([forward_states(PAIRS[name], CONTRAST, CONTRAST_CLOCK) @ (1.0, -1.0)
                    for name in SYMMETRIC], axis=1)
print("formula", outside_band(contrast, formula))

totals = np.stack([(contrast_fans[name] - PAIRS[name].stationary_intensity() / PAIRS[name].decay)
                   .sum(axis=2) for name in SYMMETRIC], axis=2)
print("1'(Z - m) against zero:", outside_band(totals, np.zeros(totals.shape[1:])))

figure = monte_carlo_figure(CONTRAST_CLOCK, contrast, formula, SYMMETRIC,
                            "From z = (1/2, 0): the mean of Z_1 - Z_2")
tau = PAIRS["self-exciting"].relaxation_time()
figure.add_scatter(x=CONTRAST_CLOCK, y=(CONTRAST @ (1.0, -1.0)) * np.exp(-CONTRAST_CLOCK / tau),
                   mode="lines", name="decay at the relaxation rate",
                   line=dict(color=INK, width=1, dash="dash"))
figure.update_xaxes(title_text="t (s)")
figure.update_yaxes(title_text="mean of Z_1(t) - Z_2(t)")
figure

The self-exciting pair returns faster than the relaxation rate and the cross-exciting pair several
times faster, while the mean of $\mathbf 1^\top(Z - m)$ is zero throughout, the Perron component
being absent.  The relaxation time bounds the decay of a mean and need not set it.  Remark 1.4.17
meets the same situation in the forecast of the mid-price, where the Perron mode drops out
altogether.

Definition 1.3.38 names two numbers: the clustering number $\bar\lambda/\beta$, the number of
events arriving within one excitation timescale, and the window count $\bar\lambda w$, the number
falling in one window of length $w$.  The first says how clustered the events are at the scale of
the excitation, the second how sparse they are at the scale of a window.  We hold the $\Gamma$ and
$\lambda^*$ of the self-exciting pair and move $\beta$ alone, with $\mu = (I - \Gamma)\lambda^*$
and $A = \beta\Gamma$: $\rho$, $\lambda^*$ and the cluster sizes are the same on every rung, and
only the timescale moves.

In [ ]:
GAMMA = PAIRS["self-exciting"].branching_matrix
LAMBDA_STAR = PAIRS["self-exciting"].stationary_intensity()
ladder = {
    decay: HawkesParams(baseline=(np.eye(2) - GAMMA) @ LAMBDA_STAR, excitation=decay * GAMMA,
                        decay=decay)
    for decay in (0.5, 2.0, 8.0, 32.0)
}
print(pd.DataFrame([
    {
        "beta (1/s)": decay,
        "rho": round(params.branching_ratio, 4),
        "lambda*": show(params.stationary_intensity()),
        "clustering number": params.stationary_intensity().sum() / decay,
        "relaxation time (s)": params.relaxation_time(),
    }
    for decay, params in ladder.items()
]).to_string(index=False))

WINDOW = 60.0
rasters = {}
for (decay, params), seed in zip(ladder.items(), (71, 72, 73, 74)):
    burn_in = 20 * params.relaxation_time()
    times, types = path(ExponentialHawkes(params, seed), burn_in + WINDOW)
    shown = times > burn_in
    name = f"β = {decay:g}, λ̄/β = {LAMBDA_STAR.sum() / decay:g}"
    rasters[name] = (times[shown] - burn_in, types[shown])
raster_figure(rasters, TYPES, "A stationary minute on each rung of the ladder")

At a small clustering number a cluster is a burst a few relaxation times long, short against the
gaps between bursts; at a large one clusters last many seconds and overlap, and the path reads as a
Poisson process with a wandering rate.  The four processes have the same rate, the same branching
ratio and the same cluster sizes.

The Fano factor of the counts in windows of length $w$, on a long stationary path of each rung,
against $w$ and against $\beta w$:

In [ ]:
SCALED_WINDOWS = np.logspace(-1.3, 2.0, 14)


def fano_factor(times, horizon, window):
    '''Variance over mean of the counts in the consecutive windows of (0, horizon].'''
    windows = int(horizon // window)
    indices = (times[times < windows * window] // window).astype(int)
    counts = np.bincount(indices, minlength=windows)
    return counts.var(ddof=1) / counts.mean()


fano = {}
for (decay, params), seed in zip(ladder.items(), (81, 82, 83, 84)):
    burn_in = 20 * params.relaxation_time()
    horizon = 100_000.0 / decay
    times, _ = path(ExponentialHawkes(params, seed), burn_in + horizon)
    times = times[times > burn_in] - burn_in
    fano[decay] = np.array([fano_factor(times, horizon, scaled / decay)
                            for scaled in SCALED_WINDOWS])

figure = make_subplots(rows=1, cols=2, shared_yaxes=True,
                       subplot_titles=("against the window w (s)", "against βw"))
for slot, (decay, values) in enumerate(fano.items()):
    colour = PALETTE[slot]
    figure.add_scatter(x=SCALED_WINDOWS / decay, y=values, mode="lines+markers",
                       name=f"β = {decay:g}", line=dict(color=colour, width=2),
                       marker=dict(size=6), row=1, col=1)
    figure.add_scatter(x=SCALED_WINDOWS, y=values, mode="lines+markers", showlegend=False,
                       line=dict(color=colour, width=2), marker=dict(size=6), row=1, col=2)
figure.update_xaxes(type="log")
figure.update_yaxes(title_text="Fano factor of the window count", row=1, col=1)
figure.update_layout(title="Self-exciting pair at four decays, with Γ and λ* fixed",
                     height=420, legend=dict(orientation="h", y=1.15, x=0))
figure

Against $w$ the four curves are one curve shifted along the axis; against $\beta w$ they coincide.
Remark 1.3.39 says that a measurement on window sums depends on $\bar\lambda/\beta$ and
$\bar\lambda w$ and not on $\beta$ or $\bar\lambda$ separately; along this ladder both numbers move,
and the Fano factor follows only their ratio, $\beta w$.  The rescaling of Remark 1.3.11 at
$c = \beta$ carries each rung to decay one and window $\beta w$, leaving a baseline $\mu/\beta$ that
differs from rung to rung, and the collapse says that the Fano factor does not see it.